In [7]:
import os
import shutil
import numpy as np
from sklearn.model_selection import train_test_split

old_base = r'C:\Users\ALYABANEYA\Desktop\first aid'
new_base =r'C:\Users\ALYABANEYA\Desktop\first-aid'
classes = ['abrasion', 'bruise', 'burn', 'cut', 'laceration', 'normal skin']

for split in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(new_base, split, cls), exist_ok=True)

print("🔄 جاري تجميع الصور وإعادة توزيعها بنسبة 70:15:15...")

for cls in classes:
    
    all_images = []
    for folder in ['train', 'val', 'test']:
        folder_path = os.path.join(old_base, folder, cls)
        if os.path.exists(folder_path):
            all_images += [os.path.join(folder_path, f) for f in os.listdir(folder_path)]
    
    
    train_imgs, temp_imgs = train_test_split(all_images, test_size=0.30, random_state=42)
    
  
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)
    
    
    for img_set, split in zip([train_imgs, val_imgs, test_imgs], ['train', 'val', 'test']):
        for img_path in img_set:
            shutil.copy(img_path, os.path.join(new_base, split, cls, os.path.basename(img_path)))

print("✅ انتهى التقسيم بنجاح!")

grand_total = 0
for split in ['train', 'val', 'test']:
    split_total = 0
    print(f"\n📊 {split.upper()}:")
    for cls in classes:
        count = len(os.listdir(os.path.join(new_base, split, cls)))
        print(f"   - {cls}: {count}")
        split_total += count
    print(f"   🔴 إجمالي الـ {split}: {split_total}")
    grand_total += split_total

print(f"\n🌎 العدد الإجمالي النهائي للداتا ست: {grand_total}")

🔄 جاري تجميع الصور وإعادة توزيعها بنسبة 70:15:15...
✅ انتهى التقسيم بنجاح!

📊 TRAIN:
   - abrasion: 1360
   - bruise: 1498
   - burn: 1262
   - cut: 1254
   - laceration: 1276
   - normal skin: 1410
   🔴 إجمالي الـ train: 8060

📊 VAL:
   - abrasion: 292
   - bruise: 321
   - burn: 270
   - cut: 269
   - laceration: 273
   - normal skin: 302
   🔴 إجمالي الـ val: 1727

📊 TEST:
   - abrasion: 292
   - bruise: 321
   - burn: 271
   - cut: 269
   - laceration: 274
   - normal skin: 303
   🔴 إجمالي الـ test: 1730

🌎 العدد الإجمالي النهائي للداتا ست: 11517


In [8]:
import os
import shutil
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

original_data_path = r'C:\Users\ALYABANEYA\Desktop\first aid\train'
output_base = r'C:\Users\ALYABANEYA\Desktop\first_aid_final'
classes = ['abrasion', 'bruise', 'burn', 'cut', 'laceration', 'normal skin']


print("🧹 جاري تنظيف الداتا والرجوع للصور الأصلية...")
all_clean_images = {cls: [] for cls in classes}

for cls in classes:
    cls_path = os.path.join(original_data_path, cls)
    if os.path.exists(cls_path):
        
        images = [f for f in os.listdir(cls_path) if not f.startswith('aug')]
        all_clean_images[cls] = [os.path.join(cls_path, img) for img in images]


print("📏 جاري التقسيم بنسبة 70:15:15...")
split_data = {'train': [], 'val': [], 'test': []}

for cls, imgs in all_clean_images.items():
    train_imgs, temp_imgs = train_test_split(imgs, test_size=0.30, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)
    
    # توزيع المسارات
    for img_list, split_name in zip([train_imgs, val_imgs, test_imgs], ['train', 'val', 'test']):
        dest_dir = os.path.join(output_base, split_name, cls)
        os.makedirs(dest_dir, exist_ok=True)
        for img in img_list:
            shutil.copy(img, os.path.join(dest_dir, os.path.basename(img)))


print("🚀 جاري موازنة مجلد التدريب فقط لـ 1711 صورة...")
target_count = 1711
datagen = ImageDataGenerator(rotation_range=40, width_shift_range=0.2, height_shift_range=0.2, 
                             shear_range=0.2, zoom_range=0.2, horizontal_flip=True, 
                             fill_mode='nearest', brightness_range=[0.8, 1.2])

for cls in classes:
    train_cls_path = os.path.join(output_base, 'train', cls)
    current_imgs = os.listdir(train_cls_path)
    num_needed = target_count - len(current_imgs)
    
    i = 0
    while i < num_needed:
        for img_name in current_imgs:
            if i >= num_needed: break
            img_path = os.path.join(train_cls_path, img_name)
            img = load_img(img_path)
            x = img_to_array(img).reshape((1,) + img_to_array(img).shape)
            
            for batch in datagen.flow(x, batch_size=1, save_to_dir=train_cls_path, 
                                     save_prefix='aug_new', save_format='jpg'):
                i += 1
                break

print(f"✅ مبروك! الداتا جاهزة في: {output_base}")

🧹 جاري تنظيف الداتا والرجوع للصور الأصلية...
📏 جاري التقسيم بنسبة 70:15:15...
🚀 جاري موازنة مجلد التدريب فقط لـ 1711 صورة...
✅ مبروك! الداتا جاهزة في: C:\Users\ALYABANEYA\Desktop\first_aid_final


In [1]:
import os

base_path = r"C:\Users\ALYABANEYA\Desktop\first.aid"

splits = ['train', 'val', 'test']

print("📊 Dataset Summary:\n")

grand_total = 0

for split in splits:
    split_total = 0
    print(f"📁 {split.upper()}:")
    
    split_path = os.path.join(base_path, split)
    
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f"   - {cls}: {count}")
            split_total += count
    
    print(f"   🔴 إجمالي الـ {split}: {split_total}\n")
    grand_total += split_total

print(f"🌎 الإجمالي الكلي للداتا: {grand_total}")

📊 Dataset Summary:

📁 TRAIN:
   - abrasion: 1665
   - bruise: 1695
   - burn: 1639
   - cut: 1628
   - laceration: 1642
   - normal skin: 1677
   🔴 إجمالي الـ train: 9946

📁 VAL:
   - abrasion: 151
   - bruise: 257
   - burn: 94
   - cut: 81
   - laceration: 95
   - normal skin: 188
   🔴 إجمالي الـ val: 866

📁 TEST:
   - abrasion: 151
   - bruise: 257
   - burn: 94
   - cut: 81
   - laceration: 95
   - normal skin: 188
   🔴 إجمالي الـ test: 866

🌎 الإجمالي الكلي للداتا: 11678
